In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import altair as alt
sys.path.append("..")
from analyse_pnb_ampm.analyse_pnb import (
    charger_sonoscores,
    statistiques_scores,
    distribution_scores,
    repartition_score_eleve_faible,
    prevalence_flags_par_groupe,
    SEUIL_SCORE_FAIBLE,
)

In [3]:
# lecture du registre des scores diagBruit (sortie de l'étape 2)
son = charger_sonoscores("../analyse_pnb_ampm/data/sonoscores_parcelles.csv")

# couleurs cohérentes sur tous les graphiques du notebook (validées CVD-safe via le skill dataviz)
COULEUR_SCORE_ELEVE = "#2a78d6"
COULEUR_SCORE_FAIBLE = "#eb6834"
GROUPE_ELEVE = f"Score élevé (>{SEUIL_SCORE_FAIBLE})"
GROUPE_FAIBLE = f"Score faible (≤{SEUIL_SCORE_FAIBLE})"
couleurs_groupe = alt.Scale(domain=[GROUPE_ELEVE, GROUPE_FAIBLE], range=[COULEUR_SCORE_ELEVE, COULEUR_SCORE_FAIBLE])

## Statistiques globales du score

In [4]:
stats = statistiques_scores(son)
print(f"Nombre de parcelles PNB analysées : {int(stats['count'])}")
print(f"Score moyen  : {stats['mean']:.2f}")
print(f"Score médian : {stats['50%']:.0f}")
print(f"Score min / max : {stats['min']:.0f} / {stats['max']:.0f}")

Nombre de parcelles PNB analysées : 21547
Score moyen  : 8.46
Score médian : 9
Score min / max : 1 / 11


## Distribution des scores

In [5]:
dist = distribution_scores(son)
chart_histogramme = alt.Chart(dist).mark_bar(color=COULEUR_SCORE_ELEVE, size=30).encode(
    x=alt.X("score:O", title="Score diagBruit"),
    y=alt.Y("nb_parcelles:Q", title="Nombre de parcelles"),
    tooltip=["score:O", "nb_parcelles:Q"],
).properties(
    title="Distribution des scores diagBruit sur les parcelles PNB",
    width=500)
chart_histogramme

alt.Chart(...)

## Score élevé vs score faible (résumé)

In [6]:
repartition = repartition_score_eleve_faible(son)
repartition["pct"] = (repartition["nb_parcelles"] / repartition["nb_parcelles"].sum() * 100)
repartition["etiquette"] = repartition["pct"].round(0).astype(int).astype(str) + "%"

base_donut = alt.Chart(repartition).encode(
    theta=alt.Theta("nb_parcelles:Q", stack=True),
    color=alt.Color("groupe:N", title="Groupe", scale=couleurs_groupe, legend=alt.Legend(orient="right")),
    tooltip=["groupe:N", "nb_parcelles:Q", alt.Tooltip("pct:Q", format=".1f", title="pct")],
)
anneau = base_donut.mark_arc(innerRadius=70, outerRadius=130)
etiquettes = base_donut.mark_text(radius=155, size=13).encode(text="etiquette:N")

chart_donut = (anneau + etiquettes).properties(
    title=f"Part des parcelles PNB à score élevé vs faible (seuil = {SEUIL_SCORE_FAIBLE})",
    width=350, height=350)
chart_donut

alt.LayerChart(...)

## Flags diagBruit selon le niveau de score

Prévalence de chacun des 7 flags dans le groupe à score faible vs le groupe à score élevé — un flag nettement sur-représenté dans le groupe "score faible" est un indice de ce qui explique un score bas malgré un bâtiment PNB confirmé.

In [7]:
flags_df = prevalence_flags_par_groupe(son)
chart_flags = alt.Chart(flags_df).mark_bar().encode(
    x=alt.X("flag:N", title="Flag", axis=alt.Axis(labelAngle=-40, labelLimit=220)),
    xOffset="groupe:N",
    y=alt.Y("prevalence_pct:Q", title="Prévalence (%)"),
    color=alt.Color("groupe:N", title="Groupe", scale=couleurs_groupe, legend=alt.Legend(orient="top-left")),
    tooltip=["flag:N", "groupe:N", alt.Tooltip("prevalence_pct:Q", format=".1f")],
).properties(
    title="Prévalence des flags diagBruit selon le niveau de score",
    width=650)
chart_flags

alt.Chart(...)